# EOD PCA-SVM Momentum Query Backtest

End-of-day PCA-SVM momentum classification on `USD-SOFR-1D-Q12STIRT` / `IMM_4xIMM_5`
using the original Matlab `pfBBGDailySVM` parameters.

**Algorithm (Matlab port):**
1. EMA-SMA spreads for short=[7, 14, 20] / long=[50, 75, 100] windows + cross-spreads → feature matrix
2. Feature normalization (StandardScaler) → PCA projection to 2 components
3. Backward-looking momentum z-score (window=10) → labels {-1, 0, +1} via z_threshold=1.75
4. SVM (RBF kernel, C=0.02, γ=0.1) trained on initial 70% of data, predictions on remaining 30%

Data sourced from `BARCHART_STIRF-RL` at `nyc_eod` frequency (Jan 2023 – Mar 2026).

In [ ]:
%load_ext autoreload
%autoreload 2

import datetime
import sys

import matplotlib.pyplot as plt
import pandas as pd
import pytz

sys.path.append("../../")

from BT.signals import pca_svm_momentum_signal, run_technical_indicator_query_backtest
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapValue import IRSwapValue
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder

NYC = pytz.timezone("America/New_York")

In [ ]:
# ── Load EOD data ────────────────────────────────────────────────────────
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
ts_builder = TimeseriesBuilder()
trade_bpv = 100_000.0

start = NYC.localize(datetime.datetime(2023, 1, 1, 18, 0))
end = NYC.localize(datetime.datetime(2026, 3, 12, 17, 0))

q = UnifiedQuery(
    curve="USD-SOFR-1D-Q12STIRT",
    tenor="IMM_4xIMM_5",
    value=UnifiedValue.IRS_RATE,
)

eod_df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=[q],
    freq="nyc_eod",
    n_jobs=12,
    routers={
        "IRS": IRSwapsTB(curve_mdp, show_tqdm=True),
    },
    ignore_cache_miss=True,
)

rate_series = pd.to_numeric(eod_df.iloc[:, 0], errors="coerce").dropna()
rate_series.name = "IMM_4xIMM_5"

print(f"Loaded {len(rate_series)} EOD bars from {rate_series.index[0]} to {rate_series.index[-1]}")
rate_series

In [ ]:
# ── PCA-SVM Momentum signal — original Matlab pfBBGDailySVM parameters ───
signal_result = pca_svm_momentum_signal(
    rate_series,
    ewma_short_spans=[7, 14, 20],       # Matlab: Short
    ewma_long_spans=[50, 75, 100],      # Matlab: Long
    n_components=2,                      # Matlab: K=2
    svm_c=0.02,                          # Matlab: C=0.02
    svm_gamma=0.1,                       # Matlab: sigma=0.1
    label_window=10,                     # Matlab: nOffset=10
    z_threshold=1.75,                    # Matlab: labelBreaks=[-inf,-1.75,1.75,inf]
    train_fraction=0.7,                  # Matlab: DataSplit=0.7
    retrain_every=None,                  # single 70/30 split (matches Matlab)
)

display(signal_result.indicator_frame.tail())
display(
    pd.DataFrame(
        {
            "rate": signal_result.raw_series,
            "desired_position": signal_result.desired_position,
            "execution_position": signal_result.execution_position,
        }
    ).tail()
)

In [ ]:
# ── Vectorized sanity check ──────────────────────────────────────────────
vectorized_position = signal_result.execution_position.fillna(0.0)
vectorized_pnl = -vectorized_position * rate_series.diff().fillna(0.0) * trade_bpv * 100.0
vectorized_cumulative_pnl = vectorized_pnl.cumsum()

fig, ax = plt.subplots(figsize=(12, 4))
vectorized_cumulative_pnl.plot(ax=ax, title="PCA-SVM Momentum — Vectorized Sanity Check (EOD)")
ax.set_ylabel("Approx PnL")
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Query-Driven Backtest

In [ ]:
def trade_query_factory(target_position, now, info):
    _ = now, info
    return IRSwapQuery(
        curve="USD-SOFR-1D-Q12STIRT",
        tenor="IMM_4xIMM_5",
        value=IRSwapValue.NPV,
        market_request={"timestamp": "now"},
        structure_kwargs={"bpv": trade_bpv * float(target_position)},
        tags=("eod_pca_svm_momentum_q12stirt",),
    )


query_backtest = run_technical_indicator_query_backtest(
    signal_result,
    trade_query_factory=trade_query_factory,
    mdp=curve_mdp,
    strategy_name="eod_pca_svm_momentum_q12stirt",
    ignore_cache_miss=True,
    show_progress=True,
)

query_backtest.metrics

In [ ]:
# ── Diagnostic plots ─────────────────────────────────────────────────────
plot_df = pd.DataFrame(
    {
        "rate": signal_result.raw_series,
        "momentum_zscore": signal_result.indicator_frame["momentum_zscore"],
        "label": signal_result.indicator_frame["label"],
        "decision_score": signal_result.indicator_frame["decision_score"],
        "pc_1": signal_result.indicator_frame["pc_1"],
        "pc_2": signal_result.indicator_frame["pc_2"],
        "execution_position": signal_result.execution_position,
    }
).dropna(subset=["rate"])

fig, axes = plt.subplots(5, 1, figsize=(14, 18), sharex=True)

plot_df["rate"].plot(ax=axes[0], title="Q12 STIRT IMM_4xIMM_5 Rate (EOD)")
plot_df["momentum_zscore"].dropna().plot(
    ax=axes[1], title="Momentum Z-Score", color="tab:orange",
)
axes[1].axhline(1.75, color="gray", linestyle="--", alpha=0.5)
axes[1].axhline(-1.75, color="gray", linestyle="--", alpha=0.5)
plot_df["execution_position"].fillna(0.0).plot(
    ax=axes[2], title="Execution Position", color="black",
)
plot_df[["pc_1", "pc_2"]].dropna().plot(ax=axes[3], title="PCA Components")
query_backtest.mtm_history.plot(ax=axes[4], title="Query-Driven MTM", color="tab:green")
axes[4].set_ylabel("PnL")

plt.tight_layout()
plt.show()

query_backtest.order_frame.tail()